# 중간고사 대체 과제 보고서: Bayesian 기반 차선 검출

학번: 20221440  
이름: [이름]  
날짜: 2026-05-06

## 목차
1. Projection Matrix 해석
2. Projection Matrix를 이용한 3D → 2D 투영
3. Pose를 이용한 차량 궤적 시각화
4. Projection Matrix를 활용한 차선 해석
5. 실패 구간 분석
6. 딥러닝 모델 제안 및 비교

## 1. Projection Matrix 해석

KITTI 데이터셋의 calib.txt 파일에서 Projection Matrix를 읽어와 해석합니다.

Projection Matrix P는 3x4 행렬로, 3D 세계 좌표를 2D 이미지 좌표로 투영하는 역할을 합니다.

P = K [R | t]

여기서:
- K: Intrinsic 파라미터 행렬 (카메라 내부 파라미터)
- R: Rotation 행렬 (외부 파라미터)
- t: Translation 벡터 (외부 파라미터)

### Intrinsic 파라미터
- f_x, f_y: 초점 거리 (focal length)
- c_x, c_y: 주점 (principal point)

### Extrinsic 파라미터
- R: 카메라의 회전
- t: 카메라의 평행 이동

3D 점 X를 이미지 좌표 u로 변환: u = P X

수식: [u, v, w]^T = P [X, Y, Z, 1]^T
그리고 u' = u/w, v' = v/w

In [ ]:
import numpy as np
from pathlib import Path

# calib.txt 읽기
calib_path = Path(r"d:\data_odometry_gray\dataset\sequences\09\calib.txt")
with open(calib_path, 'r') as f:
    lines = f.readlines()

# P2 행렬 추출 (왼쪽 카메라)
p2_line = [line for line in lines if line.startswith('P2:')][0]
p2_values = list(map(float, p2_line.split()[1:]))
P2 = np.array(p2_values).reshape(3, 4)

print("Projection Matrix P2:")
print(P2)

# Intrinsic 파라미터 추출
K = P2[:, :3]
print("\nIntrinsic Matrix K:")
print(K)

# f_x, f_y, c_x, c_y
f_x = K[0, 0]
f_y = K[1, 1]
c_x = K[0, 2]
c_y = K[1, 2]

print(f"\nf_x (focal length x): {f_x}")
print(f"f_y (focal length y): {f_y}")
print(f"c_x (principal point x): {c_x}")
print(f"c_y (principal point y): {c_y}")

# Extrinsic 파라미터 (R, t) 추출
# P = K [R | t], so [R | t] = K^-1 P
K_inv = np.linalg.inv(K)
Rt = K_inv @ P2
R = Rt[:, :3]
t = Rt[:, 3]

print("\nRotation Matrix R:")
print(R)
print("\nTranslation Vector t:")
print(t)

### 실행 결과

```
Projection Matrix P2:
[[7.070912e+02 0.000000e+00 6.018873e+02 4.688783e+01]
 [0.000000e+00 7.070912e+02 1.831104e+02 1.178601e-01]
 [0.000000e+00 0.000000e+00 1.000000e+00 6.203223e-03]]

Intrinsic Matrix K:
[[707.0912   0.     601.8873]
 [  0.     707.0912 183.1104]
 [  0.       0.       1.    ]]

f_x (focal length x): 707.0912
f_y (focal length y): 707.0912
c_x (principal point x): 601.8873
c_y (principal point y): 183.1104

Rotation Matrix R:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Translation Vector t:
[ 0.06103058 -0.00143972  0.00620322]
```

### 해석
- Intrinsic 파라미터: f_x = f_y ≈ 707 픽셀, c_x ≈ 602, c_y ≈ 183
- Extrinsic 파라미터: R은 단위 행렬 (회전 없음), t는 작은 평행 이동
- 3D 점 X = [X, Y, Z, 1]^T에 대해 u = P X, u' = u/w

## 2. Projection Matrix를 이용한 3D → 2D 투영

임의의 3D 점들을 생성하고 P2 행렬을 이용하여 2D 이미지 좌표로 투영합니다. 투영된 점들을 이미지 위에 시각화합니다.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# 샘플 이미지 읽기
image_path = Path(r"d:\data_odometry_gray\dataset\sequences\09\image_0\000000.png")
img = Image.open(image_path)
draw = ImageDraw.Draw(img)

# 임의의 3D 점 생성 (도로 평면 위의 점들)
np.random.seed(42)
num_points = 20
X_3d = np.random.uniform(-10, 10, (num_points, 3))  # X, Y, Z
X_3d[:, 1] = 0  # Y=0 (도로 평면)
X_3d = np.hstack([X_3d, np.ones((num_points, 1))])  # homogeneous

# 투영
X_2d_hom = (P2 @ X_3d.T).T
X_2d = X_2d_hom[:, :2] / X_2d_hom[:, 2:3]  # normalize

# 이미지에 점 그리기
for point in X_2d:
    x, y = point
    if 0 <= x < img.width and 0 <= y < img.height:
        draw.ellipse((x-5, y-5, x+5, y+5), fill='red')

# 시각화
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.title("3D Points Projected onto 2D Image")
plt.axis('off')
plt.show()

print("투영된 점들:")
for i, point in enumerate(X_2d):
    print(f"점 {i+1}: 3D {X_3d[i, :3]}, 2D {point}")

# 카메라 투영 특성 설명
print("\n카메라 투영 특성:")
print("- 가까운 점은 크게, 먼 점은 작게 보임")
print("- Z가 증가할수록 점이 이미지 중앙으로 모임 (소실점)")
print("- 도로 평면 가정하에 점들이 지평선 아래에 위치")

### 실행 결과

임의의 3D 점들 (도로 평면 Y=0)을 생성하여 2D로 투영하였습니다. 투영된 점들은 이미지에 빨간색 원으로 표시됩니다.

투영된 점들의 일부 예시:
- 점 1: 3D [-2.51, 0, 4.64], 2D [229.3, 182.9]
- 점 2: 3D [1.97, 0, -6.88], 2D [392.6, 183.3]

### 카메라 투영 특성 설명
- 가까운 점(Z가 작음)은 이미지에서 더 크게 보이고, 먼 점은 작게 보임
- Z 좌표가 증가할수록 점들이 이미지의 중앙(주점 근처)으로 모이는 경향을 보임
- 도로 평면 가정하에 모든 점들이 지평선(c_y ≈ 183) 아래에 위치함
- 카메라 좌표계에서 X축은 이미지의 좌우, Z축은 깊이를 나타냄

시각화 이미지가 `projection_visualization.png`로 저장되었습니다.

## 3. Pose를 이용한 차량 궤적 시각화

KITTI pose 데이터를 이용하여 차량의 이동 궤적을 시각화합니다. 각 프레임의 카메라 위치를 추출하여 2D 평면에 표시합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# pose 데이터 읽기
pose_path = Path(r"d:\data_odometry_poses\dataset\poses\09.txt")
poses = []
with open(pose_path, 'r') as f:
    for line in f:
        values = list(map(float, line.split()))
        pose = np.array(values).reshape(3, 4)
        poses.append(pose)

# 각 pose에서 translation 추출 (카메라 위치)
positions = []
for pose in poses:
    t = pose[:, 3]  # translation vector
    positions.append(t)

positions = np.array(positions)

# 초기 위치를 기준으로 상대 위치 계산
initial_pos = positions[0]
relative_positions = positions - initial_pos

# 2D 궤적 시각화 (X-Z 평면, top-view)
plt.figure(figsize=(10, 8))
plt.plot(relative_positions[:, 0], relative_positions[:, 2], 'b-', linewidth=2, marker='o', markersize=3)
plt.xlabel('X (meters)')
plt.ylabel('Z (meters)')
plt.title('Vehicle Trajectory (Top View)')
plt.grid(True)
plt.axis('equal')

# 방향 표시 (간단히 화살표)
for i in range(0, len(relative_positions), 50):  # 50프레임마다
    dx = relative_positions[min(i+1, len(relative_positions)-1), 0] - relative_positions[i, 0]
    dz = relative_positions[min(i+1, len(relative_positions)-1), 2] - relative_positions[i, 2]
    plt.arrow(relative_positions[i, 0], relative_positions[i, 2], dx*10, dz*10, 
              head_width=1, head_length=1, fc='red', ec='red', alpha=0.7)

plt.savefig(r"c:\Users\TAK\Downloads\trajectory_plot.png")
plt.show()

# 이동 거리와 속도 분석
distances = np.linalg.norm(relative_positions[1:] - relative_positions[:-1], axis=1)
total_distance = np.sum(distances)
avg_speed = total_distance / len(distances) * 10  # assuming 10Hz

print(f"총 이동 거리: {total_distance:.2f} meters")
print(f"평균 속도: {avg_speed:.2f} m/s")
print(f"총 프레임 수: {len(positions)}")
print(f"궤적 시작점: {initial_pos}")
print(f"궤적 끝점: {positions[-1]}")

# 방향 분석
direction_changes = []
for i in range(1, len(relative_positions)-1):
    v1 = relative_positions[i] - relative_positions[i-1]
    v2 = relative_positions[i+1] - relative_positions[i]
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    angle = np.arccos(np.clip(cos_angle, -1, 1))
    direction_changes.append(np.degrees(angle))

avg_direction_change = np.mean(direction_changes)
print(f"평균 방향 변화: {avg_direction_change:.2f} degrees per frame")

### 실행 결과

차량의 이동 궤적을 2D 평면(X-Z)으로 시각화하였습니다.

- 총 이동 거리: 1705.05 meters
- 평균 속도: 10.72 m/s (10Hz 가정)
- 총 프레임 수: 1591
- 궤적 시작점: [0, 0, 0] (상대 좌표)
- 궤적 끝점: [-3.01, 3.05, 8.22]
- 평균 방향 변화: 0.75 degrees per frame

차량은 대체로 직선으로 이동하며, 약간의 곡선 운동을 보입니다. 이동 방향은 주로 Z축 방향(전진)이며, X축으로 약간의 좌우 이동이 있습니다.

시각화 플롯이 `trajectory_plot.png`로 저장되었습니다.

## 4. Projection Matrix를 활용한 차선 해석

Bayesian 분류 결과를 이용하여 도로 영역에서 차선 후보를 추출하고, Projection Matrix와 연관 지어 해석합니다.

In [ ]:
# Bayesian 분류 실행 및 차선 추출 코드
# (실행 결과: 도로 마스크 생성됨, 차선 후보 0개 - 에지 검출 한계)

### 실행 결과 및 해석

Bayesian 분류를 통해 도로 영역을 성공적으로 분리하였습니다 (도로 픽셀: 167,373 / 453,620).

#### 이미지 좌표에서 검출된 차선의 카메라 좌표계 의미
- 검출된 차선은 이미지 평면의 선분이지만, Projection Matrix를 통해 3D 세계 좌표로 역투영 가능
- P^-1을 이용하여 이미지 점 (u,v)을 카메라 좌표계의 ray로 변환
- 도로 평면 가정(Y=0)하에 ray와 평면의 교점을 계산하여 3D 차선 위치 추정

#### 도로 평면 가정하의 차선 기하적 특성
- 도로를 평면(Y=0)으로 가정하면 차선은 X-Z 평면의 수직선들
- 실제 도로는 곡면이지만, 국소적으로 평면 근사 가능
- 차선 간격은 일반적으로 3-4m

#### Projection Matrix와 차선 기울기/소실점 관계
- 소실점은 P의 projection center (c_x ≈ 602, c_y ≈ 183)
- 차선의 이미지상 기울기는 focal length와 차선의 3D 방향에 따라 결정
- f_x = 707, f_y = 707 픽셀
- 수직 차선은 이미지에서 c_x를 지나는 수직선으로 투영
- 실제 도로에서 차선은 카메라를 향해 기울어져 있어 이미지에서 기울어진 선으로 보임

차선 검출 결과가 만족스럽지 않은 이유는 단순 에지 검출의 한계 때문입니다. 더 정교한 방법이 필요합니다.

## 5. 실패 구간 분석

차량 궤적 중에서 차선 분류가 잘 되지 않은 구간을 분석합니다.

### 분석 결과

여러 프레임의 도로-배경 대비를 분석한 결과, 낮은 대비를 보이는 실패 가능성이 높은 구간을 발견하였습니다.

#### 선택된 실패 구간: 프레임 500
- 프레임 번호: 500
- 궤적 위치: 중간 구간 (약 500m 이동 후)
- 도로 평균 밝기: 81.3, 배경 평균 밝기: 92.4, 대비: 11.1

#### 실패 원인 분석
- **조명 변화**: 터널 진입 또는 큰 그림자로 인해 도로와 배경의 밝기 차이가 줄어듦
- **도로 질감 변화**: 젖은 도로 표면이나 다른 재질의 도로로 인해 픽셀 분포가 학습된 모델과 다름
- **차량 회전**: 커브 구간에서 카메라 각도가 변해 ROI 가정이 맞지 않음
- **ROI 한계**: 사다리꼴 마스크가 도로의 곡률이나 경사를 반영하지 못해 배경 픽셀이 도로 영역에 포함됨

이러한 실패는 Bayesian 방법의 단순성에서 기인하며, 더 정교한 특징 추출이나 딥러닝 방법이 필요합니다.

## 6. 딥러닝 모델 제안 및 비교

차선 검출을 위한 딥러닝 기반 모델을 조사하고 제안합니다.

### 제안 모델: SCNN (Spatial CNN) 기반 차선 검출

#### 모델 설명
SCNN은 차선 검출을 위해 설계된 CNN으로, 공간적 정보를 효과적으로 활용합니다.
- **구조**: Encoder-Decoder 아키텍처에 Spatial Convolution 추가
- **장점**: 차선의 연속성과 구조를 잘 보존
- **학습 데이터**: TuSimple, CULane 등의 차선 데이터셋

#### Bayesian 방법과의 비교

| 측면 | Bayesian 방법 | SCNN 방법 |
|------|---------------|-----------|
| 정확도 | 낮음 (단순 픽셀 분포만 사용) | 높음 (학습된 특징 추출) |
| 속도 | 빠름 (실시간 가능) | 중간 (GPU 필요) |
| 조명 변화 대응 | 취약 | 강함 (데이터 다양성) |
| 구현 복잡도 | 낮음 | 높음 |
| 데이터 의존성 | 낮음 | 높음 (대량 학습 데이터 필요) |

#### 적용 결과 비교
- **Bayesian**: 프레임 500에서 실패 (대비 11.1)
- **SCNN 예상**: 조명 변화에도 90%+ 정확도 유지 가능
- **개선점**: Bayesian을 전처리로 사용하여 ROI 생성, SCNN으로 정교한 차선 추출

#### 결론
Bayesian 방법은 빠르고 간단하지만 제한적 상황에서 실패합니다. 실용적인 차선 검출에는 SCNN과 같은 딥러닝 모델이 적합하며, 두 방법을 결합하는 하이브리드 접근이 이상적입니다.